# P3 · Customer Segmentation — Strategy Mapping

**Project:** P3 · Coffra Customer Segmentation
**Author:** Sebastian Kradyel
**Date:** April 2026
**Notebook:** 05_segment_to_strategy.ipynb

---

## Purpose

This notebook bridges analytical segmentation (notebooks 02–04) to Coffra-specific marketing strategy. It maps both rule-based segments and ML clusters to the two Coffra personas (Connoisseur, Daily Ritualist), defines retention/acquisition tactics per segment, and computes financial impact projections.

## Why this notebook matters

A segmentation model that does not produce action is academic. The deliverable here is a **strategic playbook** that a marketing manager could execute Monday morning. Each segment gets:
1. **Persona alignment** (which Coffra persona does this segment most likely correspond to)
2. **Recommended channel + cadence**
3. **Specific marketing campaign** referencing P1 emails and HubSpot workflows
4. **Expected business impact** with explicit assumptions

## Note on transferability

This analysis uses the Online Retail II dataset (UK gift retailer) but maps insights to Coffra. The mapping is **structural**, not literal — RFM dynamics are universal across e-commerce verticals. A Champion in retail behaves like a Champion in coffee subscription: high-value, frequent, recent. Strategies transfer directly.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

COFFRA_BROWN = '#3E2723'
COFFRA_BROWN_LIGHT = '#6D4C41'
COFFRA_CREAM = '#EFEBE9'
COFFRA_PALETTE = [COFFRA_BROWN, COFFRA_BROWN_LIGHT, '#A1887F', '#BCAAA4', '#D7CCC8', '#8D6E63', '#5D4037']

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

INPUT_PATH = Path('../data/processed/rfm_clustered.parquet')
OUTPUT_DIR = Path('../data/processed')

In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(f'Loaded: {len(df):,} customers')
df.head()

## 2. Persona Alignment Heuristic

Coffra has two personas (defined in P1):
- **Connoisseur (Andrei archetype):** Technical, brand-loyal, premium spender. High frequency on smaller basket per visit (single-origin coffee enthusiasts buy specific products often).
- **Daily Ritualist (Bianca archetype):** Habitual, lifestyle-driven, moderate frequency, café-goer. Lower-frequency but consistent purchases (Coffra Pass subscribers, monthly bean buyers).

We don't have explicit persona data in this dataset, but we can **infer probable persona** using RFM signature:

**Connoisseur signature:**
- High frequency (10+ invoices)
- High monetary (top quintile)
- High recency (active in last 90 days)

**Daily Ritualist signature:**
- Moderate frequency (3–10 invoices)
- Mid monetary (middle 3 quintiles)
- Moderate recency (60-180 days)

**Casual/Unknown:** Everyone else.

In [ ]:
def assign_persona(row):
    r, f, m_score = row['Recency'], row['Frequency'], row['M_Score']
    
    # Connoisseur: high engagement on all dimensions
    if f >= 10 and m_score >= 4 and r <= 90:
        return 'Connoisseur (probable)'
    
    # Daily Ritualist: moderate engagement, sustained over time
    if 3 <= f <= 10 and 2 <= m_score <= 4 and r <= 180:
        return 'Daily Ritualist (probable)'
    
    # Default: not enough signal to assign
    return 'Unaligned'

df['Probable_Persona'] = df.apply(assign_persona, axis=1)

persona_dist = df['Probable_Persona'].value_counts()
print('Probable persona distribution:')
print(persona_dist)
print(f'\nNote: "Unaligned" is expected — many customers don\'t fit either profile cleanly.')
print('In production with real Coffra data, persona assignment would be explicit (signup form / preference survey).')

In [ ]:
# Cross-tab: persona vs cluster
persona_cluster = pd.crosstab(df['Probable_Persona'], df['kmeans_label'], margins=True)
print('Probable persona × K-Means cluster:')
print(persona_cluster)

## 3. Strategic Action Mapping

Each combination of segment and persona deserves a specific marketing approach. Below is the strategic playbook.

In [ ]:
strategy_playbook = pd.DataFrame([
    {
        'Segment': 'Champions',
        'Persona alignment': 'Connoisseur (primary), Daily Ritualist (secondary)',
        'Action': 'Retention + advocacy program',
        'Channel': 'Personal email from Sebastian (Roaster) + private events',
        'Cadence': 'Monthly (low-volume, high-touch)',
        'P1 reference': 'E5 Comparison Test (Connoisseur), E10 Community Invitation (DR)',
        'Expected impact': 'Maintain 90%+ retention, +20% LTV via referrals'
    },
    {
        'Segment': 'Loyal Customers',
        'Persona alignment': 'Connoisseur (primary)',
        'Action': 'Subscription upsell + loyalty rewards',
        'Channel': 'Email automation (HubSpot workflow)',
        'Cadence': 'Bi-weekly with seasonal campaigns',
        'P1 reference': 'E4 Subscription Pitch (Connoisseur sequence)',
        'Expected impact': '15-25% subscription conversion'
    },
    {
        'Segment': 'Potential Loyalists',
        'Persona alignment': 'Connoisseur (probable, early stage)',
        'Action': 'Accelerated nurture to reach Loyal status',
        'Channel': 'Email automation + SMS triggers',
        'Cadence': 'Weekly for 60 days, then bi-weekly',
        'P1 reference': 'Full Connoisseur 5-email sequence',
        'Expected impact': '30-40% promote to Loyal within 90 days'
    },
    {
        'Segment': 'Recent Customers',
        'Persona alignment': 'Either persona (insufficient signal)',
        'Action': 'Onboarding + persona discovery',
        'Channel': 'Email automation + welcome survey',
        'Cadence': '4 emails over 14 days',
        'P1 reference': 'E1 Welcome (both personas)',
        'Expected impact': 'Survey response 25-35% → enables persona-specific routing'
    },
    {
        'Segment': 'Promising',
        'Persona alignment': 'Daily Ritualist (probable)',
        'Action': 'Coffra Pass introduction',
        'Channel': 'Email + targeted Instagram ads',
        'Cadence': 'Bi-weekly with social media touchpoints',
        'P1 reference': 'E4 Coffra Pass (Daily Ritualist sequence)',
        'Expected impact': '10-15% Coffra Pass conversion'
    },
    {
        'Segment': 'Customers Needing Attention',
        'Persona alignment': 'Mostly Daily Ritualist',
        'Action': 'Re-engagement with personalization',
        'Channel': 'Email with last-purchase reminder',
        'Cadence': '3-touch campaign over 21 days',
        'P1 reference': 'Adapt cart recovery emails (T+1h, T+24h, T+72h)',
        'Expected impact': '20% reactivation rate'
    },
    {
        'Segment': 'Cannot Lose Them',
        'Persona alignment': 'Connoisseur (high-value, slipping)',
        'Action': 'High-priority win-back from Sebastian directly',
        'Channel': 'Personal email + phone call (if churn risk high)',
        'Cadence': '1-on-1, immediate',
        'P1 reference': 'Custom message — no template applies',
        'Expected impact': '40-50% win-back, justifies high-touch cost'
    },
    {
        'Segment': 'About to Sleep',
        'Persona alignment': 'Daily Ritualist (drift mode)',
        'Action': 'Light reminder of Coffra value',
        'Channel': 'Email automation',
        'Cadence': '2 emails over 30 days',
        'P1 reference': 'E3 Three Rituals (Daily Ritualist, lifestyle anchoring)',
        'Expected impact': '15% prevented churn'
    },
    {
        'Segment': 'At Risk',
        'Persona alignment': 'Mixed (mostly mid-value persona)',
        'Action': 'Win-back with discount (asymmetric per persona)',
        'Channel': 'Email + retargeting ads',
        'Cadence': '2-3 emails over 14 days',
        'P1 reference': 'E2 cart recovery template (Quick thought, free shipping)',
        'Expected impact': '12-18% win-back'
    },
    {
        'Segment': 'Hibernating',
        'Persona alignment': 'Unknown (long-dormant)',
        'Action': 'Last-chance win-back; then suppress from active list',
        'Channel': 'Single email, no retry',
        'Cadence': '1 email; if no engagement, exclude from sends',
        'P1 reference': 'Adapt E5 Comparison Test (final opportunity)',
        'Expected impact': '5-8% reactivation; main goal is list hygiene'
    },
    {
        'Segment': 'Lost',
        'Persona alignment': 'Inactive (any)',
        'Action': 'Suppress from email; revisit in 6 months for fresh campaign',
        'Channel': 'None active',
        'Cadence': 'No outbound (saves sender reputation)',
        'P1 reference': 'No active campaign',
        'Expected impact': 'List hygiene; 1-2% return rate organically'
    },
])

print(f'Strategic playbook covers {len(strategy_playbook)} segments.')
strategy_playbook

## 4. Financial Impact Projection

Quantify expected lift from segment-specific campaigns vs status quo (untargeted broadcast).

**Method:** For each segment, multiply the customer count by the expected per-customer lift in monthly revenue.

**Assumptions** (clearly labeled):
- Status quo retention: 75% monthly (industry e-commerce avg)
- Segment-specific lifts based on industry benchmarks for targeted campaigns (Klaviyo, Bloomreach 2024 reports)
- Coffra avg order value: £20 (Pass = £40/month, single beans = £15/order, ~mid-range)
- All values are **scenario projections**, not measurements

In [ ]:
# Define expected lift assumptions per segment
lift_assumptions = {
    'Champions': {'retention_lift_pct': 0.05, 'aov_uplift_pct': 0.10, 'avg_monthly_orders': 2.0},
    'Loyal Customers': {'retention_lift_pct': 0.10, 'aov_uplift_pct': 0.15, 'avg_monthly_orders': 1.5},
    'Potential Loyalists': {'retention_lift_pct': 0.20, 'aov_uplift_pct': 0.05, 'avg_monthly_orders': 1.0},
    'Recent Customers': {'retention_lift_pct': 0.15, 'aov_uplift_pct': 0.0, 'avg_monthly_orders': 0.5},
    'Promising': {'retention_lift_pct': 0.12, 'aov_uplift_pct': 0.05, 'avg_monthly_orders': 0.5},
    'Customers Needing Attention': {'retention_lift_pct': 0.18, 'aov_uplift_pct': 0.0, 'avg_monthly_orders': 0.3},
    'Cannot Lose Them': {'retention_lift_pct': 0.40, 'aov_uplift_pct': 0.0, 'avg_monthly_orders': 1.5},
    'About to Sleep': {'retention_lift_pct': 0.10, 'aov_uplift_pct': 0.0, 'avg_monthly_orders': 0.3},
    'At Risk': {'retention_lift_pct': 0.15, 'aov_uplift_pct': 0.0, 'avg_monthly_orders': 0.5},
    'Hibernating': {'retention_lift_pct': 0.05, 'aov_uplift_pct': 0.0, 'avg_monthly_orders': 0.1},
    'Lost': {'retention_lift_pct': 0.01, 'aov_uplift_pct': 0.0, 'avg_monthly_orders': 0.0},
}

AOV = 20.0  # Average order value, GBP — Coffra assumption
MONTHS_HORIZON = 12  # Annual projection

# Compute projection per segment
impact = []
for seg, params in lift_assumptions.items():
    customers = (df['Segment'] == seg).sum()
    if customers == 0:
        continue
    
    monthly_baseline = customers * params['avg_monthly_orders'] * AOV
    monthly_with_lift = customers * params['avg_monthly_orders'] * AOV * (
        1 + params['retention_lift_pct'] + params['aov_uplift_pct']
    )
    monthly_uplift = monthly_with_lift - monthly_baseline
    annual_uplift = monthly_uplift * MONTHS_HORIZON
    
    impact.append({
        'Segment': seg,
        'Customers': int(customers),
        'Lift_pct': f"{(params['retention_lift_pct'] + params['aov_uplift_pct'])*100:.0f}%",
        'Monthly_baseline_GBP': round(monthly_baseline, 0),
        'Monthly_uplift_GBP': round(monthly_uplift, 0),
        'Annual_uplift_GBP': round(annual_uplift, 0),
    })

impact_df = pd.DataFrame(impact)
impact_df = impact_df.sort_values('Annual_uplift_GBP', ascending=False)
print('Annual revenue uplift projection by segment:')
print(impact_df.to_string(index=False))

total_annual_uplift = impact_df['Annual_uplift_GBP'].sum()
total_baseline = impact_df['Monthly_baseline_GBP'].sum() * MONTHS_HORIZON
print(f'\nTotal projected annual uplift: £{total_annual_uplift:,.0f}')
print(f'Total annual baseline (untargeted): £{total_baseline:,.0f}')
print(f'Relative lift: +{total_annual_uplift/total_baseline*100:.1f}%')

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Top: annual uplift
impact_sorted = impact_df.sort_values('Annual_uplift_GBP', ascending=True)
axes[0].barh(impact_sorted['Segment'], impact_sorted['Annual_uplift_GBP'],
             color=COFFRA_BROWN, alpha=0.85)
axes[0].set_title('Annual Revenue Uplift Projection (£) by Segment')
axes[0].set_xlabel('Projected uplift (£)')
axes[0].grid(alpha=0.3, axis='x')

# Bottom: customers per segment (for context)
axes[1].barh(impact_sorted['Segment'], impact_sorted['Customers'],
             color=COFFRA_BROWN_LIGHT, alpha=0.85)
axes[1].set_title('Number of Customers per Segment')
axes[1].set_xlabel('Customers')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

**Insight:** The financial story is clear — a few high-leverage segments dominate the uplift potential:
- **Champions and Loyal Customers** generate the largest absolute uplift despite being a small share of customers, because each customer is high-value and even modest lifts compound.
- **Cannot Lose Them** segment justifies high-touch (personal email, phone) because the per-customer impact is massive.
- **Lost** segment has minimal projected uplift, validating the suppression strategy.

## 5. Per-Persona Action Summary

Aggregate strategy by probable persona for executive summary.

In [ ]:
persona_summary = df.groupby('Probable_Persona').agg(
    customers=('Customer ID', 'count'),
    avg_recency=('Recency', 'mean'),
    avg_frequency=('Frequency', 'mean'),
    avg_monetary=('Monetary', 'mean'),
    total_revenue=('Monetary', 'sum'),
).round(2)

persona_summary['pct_customers'] = (persona_summary['customers'] / persona_summary['customers'].sum() * 100).round(1)
persona_summary['pct_revenue'] = (persona_summary['total_revenue'] / persona_summary['total_revenue'].sum() * 100).round(1)

print('Customer base by probable persona:')
print(persona_summary)

In [ ]:
# Visualization: persona contribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].pie(persona_summary['customers'], labels=persona_summary.index,
            autopct='%1.1f%%', colors=COFFRA_PALETTE[:len(persona_summary)],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Customer Distribution by Probable Persona')

axes[1].pie(persona_summary['total_revenue'], labels=persona_summary.index,
            autopct='%1.1f%%', colors=COFFRA_PALETTE[:len(persona_summary)],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Revenue Distribution by Probable Persona')

plt.tight_layout()
plt.show()

## 6. Coffra Implementation Roadmap

How to deploy this segmentation in production.

In [ ]:
roadmap = pd.DataFrame([
    {
        'Phase': '1. Data Foundation (Weeks 1-2)',
        'Tasks': 'Set up purchase tracking in HubSpot. Sync orders from Shopify. Establish Customer ID consistency across systems.',
        'Owner': 'Sebastian + dev partner'
    },
    {
        'Phase': '2. RFM Pipeline (Weeks 3-4)',
        'Tasks': 'Adapt notebook 03 (RFM scoring) to run on Coffra Shopify data. Schedule weekly refresh (cron job). Persist RFM scores as HubSpot custom properties.',
        'Owner': 'Marketing + Engineering'
    },
    {
        'Phase': '3. Segment Workflows (Weeks 5-6)',
        'Tasks': 'Configure HubSpot workflows triggered by segment property changes. Map each segment to email sequences from P1. Set up suppression rules for Hibernating/Lost.',
        'Owner': 'Marketing'
    },
    {
        'Phase': '4. Persona Survey (Week 6)',
        'Tasks': 'Add 1-question survey to Recent Customers email: "Are you here for the daily coffee experience or specialty single-origin beans?" Use response to populate persona property explicitly.',
        'Owner': 'Marketing'
    },
    {
        'Phase': '5. Measurement (Ongoing)',
        'Tasks': 'Monthly dashboard: segment migration matrix (who moved between segments), retention rate per segment, revenue per segment vs forecast.',
        'Owner': 'Marketing'
    },
    {
        'Phase': '6. Iteration (Quarterly)',
        'Tasks': 'Re-run clustering with new data. Validate segment definitions still hold. Adjust action playbook based on observed lifts.',
        'Owner': 'Marketing + Analytics'
    },
])

print('Coffra implementation roadmap:')
print(roadmap.to_string(index=False))

## 7. Save Strategic Outputs

Save all strategic artifacts for the case study and dashboard.

In [ ]:
# Save the playbook
strategy_playbook.to_csv(OUTPUT_DIR / 'strategy_playbook.csv', index=False)
print(f'Saved: {OUTPUT_DIR / "strategy_playbook.csv"}')

# Save the impact projections
impact_df.to_csv(OUTPUT_DIR / 'impact_projections.csv', index=False)
print(f'Saved: {OUTPUT_DIR / "impact_projections.csv"}')

# Save the final per-customer file with persona overlay
df.to_parquet(OUTPUT_DIR / 'final_segmentation.parquet', index=False)
print(f'Saved: {OUTPUT_DIR / "final_segmentation.parquet"}')

# Save unified summary JSON for dashboard
import json

final_summary = {
    'analysis_date': pd.Timestamp.now().isoformat(),
    'total_customers': int(len(df)),
    'total_revenue_observed_gbp': round(float(df['Monetary'].sum()), 2),
    'projected_annual_uplift_gbp': round(float(impact_df['Annual_uplift_GBP'].sum()), 2),
    'projected_lift_pct': round(float(total_annual_uplift / total_baseline * 100), 2),
    'segment_counts': df['Segment'].value_counts().to_dict(),
    'cluster_counts': df['kmeans_label'].value_counts().to_dict(),
    'persona_counts': df['Probable_Persona'].value_counts().to_dict(),
    'top_3_uplift_segments': impact_df.head(3)[['Segment', 'Annual_uplift_GBP']].to_dict('records'),
    'methodology_note': (
        'Lift assumptions are scenario-based, anchored to industry benchmarks (Klaviyo, '
        'Bloomreach 2024). They represent reasonable projections for differentiated '
        'segment campaigns vs. untargeted broadcast. Real Coffra deployment would measure '
        'actual lifts via A/B testing per segment.'
    ),
}

with open(OUTPUT_DIR / 'segmentation_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=2)

print(f'Saved: {OUTPUT_DIR / "segmentation_summary.json"}')
print('\nP3 segmentation analysis complete.')

---

## Summary

**Strategic deliverables:**
- 11-segment playbook with action, channel, cadence, P1 reference per segment.
- Persona alignment heuristic mapping RFM signature to Connoisseur / Daily Ritualist.
- Annual revenue uplift projection: total + per-segment breakdown.
- Coffra implementation roadmap (6 phases over ~10 weeks).

**Key strategic insights:**
1. Champions and Loyal Customers (~15-20% of base) drive disproportionate uplift potential because lifts compound on already-high revenue.
2. 'Cannot Lose Them' justifies high-touch (personal email, phone) — single-customer ROI is exceptional.
3. Lost segment should be suppressed, not retargeted — protects sender reputation and saves cost.
4. Persona alignment is best done **explicitly** via signup survey, not inferred from behavior alone.

**Honest disclosure:**
- All uplift projections are scenario-based, anchored to published benchmarks.
- No A/B testing was performed (would require live Coffra customer base).
- Persona inference is best-effort heuristic — production system needs explicit persona signals.

**Output artifacts (for dashboard and case study):**
- `strategy_playbook.csv` — 11-segment action playbook
- `impact_projections.csv` — financial uplift table
- `final_segmentation.parquet` — per-customer enriched data
- `segmentation_summary.json` — unified KPIs

---

## Versioning

| Version | Date | Changes |
|---------|------|---------|
| **v1.0** | **April 26, 2026** | Initial strategic mapping with 11-segment playbook, financial impact projection, and Coffra implementation roadmap. |